In [ ]:
# ==========================================
# MODULO 1: REGISTRO DE USUARIOS
# ==========================================

from google.colab import drive
import csv
import os
import hashlib
import secrets
from datetime import date
from getpass import getpass

# Conectamos Google Drive

drive.mount('/content/drive')

# Definimos donde guardaremos los datos

carpeta = "/content/drive/MyDrive/MoniAppsFinal"

os.makedirs(carpeta, exist_ok=True)

# Archivo donde guardaremos los usuarios

archivo_usuarios = carpeta + "/usuarios.csv"

archivo_saldo_inicial = carpeta + "/saldo_inicial.csv"

print("MoniApps esta preparado correctamente.")

In [ ]:
# ==========================================
# MODULO 7: REGISTRO DE INGRESOS Y GASTOS
# ==========================================

# Archivo donde guardaremos los movimientos

archivo_movimientos = carpeta + "/movimientos.csv"


def registrar_movimiento(usuario_activo):

    # Verificamos que exista una sesion activa

    if usuario_activo is None:
        print("Primero debes iniciar sesion.")
        return

    print("\nREGISTRO DE MOVIMIENTOS")
    print("-----------------------")

    # Seleccionamos el tipo de movimiento

    print("1. Registrar ingreso")
    print("2. Registrar gasto")

    opcion = input("Selecciona una opcion: ")

    if opcion == "1":
        tipo = "Ingreso"

    elif opcion == "2":
        tipo = "Gasto"

    else:
        print("Opcion no valida.")
        return

    # Consultamos las categorias del usuario

    with open(archivo_categorias, "r", encoding="utf-8") as archivo:

        lector = csv.DictReader(archivo)

        categorias = [
            fila["categoria"]
            for fila in lector
            if fila["id_usuario"] == str(usuario_activo["id_usuario"])
            and fila["tipo"] == tipo
        ]

    if not categorias:
        print("No tienes categorias configuradas para este movimiento.")
        return

    # Mostramos las categorias disponibles

    print("\nCATEGORIAS DISPONIBLES")

    for i, categoria in enumerate(categorias, start=1):
        print(i, "-", categoria)

    # Seleccionamos una categoria

    try:
        seleccion = int(input("Selecciona una categoria: "))

        if seleccion < 1 or seleccion > len(categorias):
            print("Seleccion no valida.")
            return

        categoria_seleccionada = categorias[seleccion - 1]

    except ValueError:
        print("Debes ingresar un numero.")
        return

    # Solicitamos la descripcion

    descripcion = input("Descripcion del movimiento: ").strip()

    if descripcion == "":
        print("Debes escribir una descripcion.")
        return

    # Solicitamos el valor

    try:
        valor = float(input("Valor del movimiento: $"))

        if valor <= 0:
            print("El valor debe ser mayor que cero.")
            return

    except ValueError:
        print("Debes ingresar un valor numerico.")
        return

    # Registramos la fecha actual

    fecha_movimiento = date.today().isoformat()

    # Preparamos la informacion del movimiento

    nuevo_movimiento = {
        "id_usuario": usuario_activo["id_usuario"],
        "tipo": tipo,
        "categoria": categoria_seleccionada,
        "descripcion": descripcion,
        "valor": valor,
        "fecha": fecha_movimiento
    }

    # Definimos los campos del archivo

    campos = [
        "id_usuario",
        "tipo",
        "categoria",
        "descripcion",
        "valor",
        "fecha"
    ]

    # Guardamos el movimiento en CSV

    existe = os.path.exists(archivo_movimientos)

    with open(archivo_movimientos, "a", newline="", encoding="utf-8") as archivo:

        escritor = csv.DictWriter(archivo, fieldnames=campos)

        if not existe or os.path.getsize(archivo_movimientos) == 0:
            escritor.writeheader()

        escritor.writerow(nuevo_movimiento)

    # Confirmamos el registro

    print("\n¡MOVIMIENTO REGISTRADO CORRECTAMENTE!")
    print("------------------------------------")
    print("Tipo:", tipo)
    print("Categoria:", categoria_seleccionada)
    print("Descripcion:", descripcion)
    print("Valor: $", valor)
    print("Fecha:", fecha_movimiento)

In [ ]:
# ==========================================
# MODULO 9: MENU PRINCIPAL
# ==========================================

def menu_principal(usuario_activo):

    # Verificamos que exista una sesion activa

    if usuario_activo is None:
        print("Debes iniciar sesion para continuar.")
        return

    print("\n===================================")
    print("     MoniApps")
    print("===================================")
    print("Bienvenido(a):", usuario_activo["nombre"])

    # Creamos un ciclo para mantener el menu activo

    while True:

        print("1. Configurar categorias")
        print("2. Configurar presupuesto")
        print("3. Configurar saldo inicial")
        print("4. Registrar ingresos y gastos")
        print("5. Consultar balance financiero")
        print("6. Cerrar sesion")

        opcion = input("\nSelecciona una opcion: ")

        # Ejecutamos la funcion correspondiente

        if opcion == "1":
            configurar_categorias(usuario_activo)

        elif opcion == "2":
          configurar_presupuesto(usuario_activo)

        elif opcion == "3":
            configurar_saldo_inicial(usuario_activo)

        elif opcion == "4":
            registrar_movimiento(usuario_activo)

        elif opcion == "5":
            consultar_balance(usuario_activo)

        elif opcion == "6":
            print("\nSesion cerrada correctamente.")
            print("¡Gracias por utilizar MoniApps!")
            break

        else:

            print("\nOpcion no valida. Intenta nuevamente.")